# 9장 Plotly 기초: 탐색하는 차트 만들기

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 위에서부터 차례로 실행합니다.

## 9.1 정적 그림에서 탐색으로

In [ ]:
# P05 | Plotly 준비: 라이브러리를 불러오고 기본 모양을 정합니다.
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

print('Plotly:', plotly.__version__)
pio.templates.default = 'plotly_white'               # 흰 배경·옅은 격자
px.defaults.line_dash_sequence = ['solid', 'dash']   # 둘째 그룹은 파선
px.defaults.symbol_sequence = ['circle', 'x']        # 둘째 그룹은 X 점

In [ ]:
# C01 | 공통 데이터: 한 행은 한 달·한 지점이고, 광고비 열을 더합니다.
df = pd.DataFrame({
    'month': [1, 2, 3, 4] * 2,
    'branch': ['A'] * 4 + ['B'] * 4,
    'sales': [100, 120, 150, 130, 70, 90, 100, 120],
    'profit': [10, 15, 25, 20, 7, 9, 12, 18],
    'marketing': [10, 12, 15, 13, 7, 9, 11, 12],
})
LABELS = {'month': '월', 'branch': '지점', 'sales': '매출 (백만원)',
          'profit': '영업이익 (백만원)', 'marketing': '광고비 (백만원)'}
branch_colors = {'A': '#0072B2', 'B': '#D55E00'}   # 지점 색(4장과 같음)
print('행 수:', len(df), '| 월·지점 중복:',
      df.duplicated(['month', 'branch']).any())
print(df.tail(3))

In [ ]:
# C02 | 첫 Plotly 차트: 4장의 열 연결을 Plotly Express로 옮깁니다.
fig_line = px.line(df, x='month', y='sales', color='branch',
                   line_dash='branch', markers=True, labels=LABELS,
                   color_discrete_map=branch_colors,
                   title='1~4월 지점별 매출')
fig_line.update_xaxes(dtick=1)          # 눈금 간격 1(1~4월)
fig_line.update_yaxes(range=[0, 160])
fig_line.show()

### 직접 해 보기

**W9-01** 매출 대신 영업이익 선 그래프를 만듭니다.

1. C02를 새 이름(예: `fig_profit`)으로 복사해 `y='profit'`으로 바꾸고, 제목을 영업이익에 맞게 고칩니다.
2. Y축 범위를 0~30으로 맞춥니다. Y축 라벨의 단위는 `LABELS`가 정합니다.
3. 툴팁으로 A지점 영업이익이 가장 높은 월과 값을 읽고, `df`의 같은 행과 대조합니다.

**완료 기준**: 두 선에 점이 4개씩이고 값이 영업이익이며, 제목·범위·단위가 맞고, 읽은 최고 월과 값을 원자료와 대조했다.

In [ ]:
# W9-01 — 여기에서 직접 해 봅니다.

## 9.2 산점도·툴팁·평균 집계

In [ ]:
# C03 | 산점도와 툴팁: 축에 없는 월과 영업이익을 툴팁에 더합니다.
fig_scatter = px.scatter(df, x='marketing', y='sales', color='branch',
                         symbol='branch', hover_name='branch',
                         hover_data={'month': True, 'profit': ':.0f'},
                         labels=LABELS, color_discrete_map=branch_colors,
                         title='광고비와 매출: 8개 관측')
fig_scatter.update_traces(marker_size=12)
fig_scatter.update_xaxes(range=[6, 19.5])     # 오른쪽에 툴팁 자리
fig_scatter.show()

In [ ]:
# C04 | 진단용: 표의 행 8개를 막대에 그대로 넣어 봅니다.
raw_bar = px.bar(df, x='branch', y='sales', color='branch',
                 hover_data=['month'], labels=LABELS,
                 color_discrete_map=branch_colors,
                 title='진단용: 원래 행 8개를 그대로 넣은 막대')
raw_bar.update_yaxes(range=[0, 560])
raw_bar.show()
print('합계:', df.groupby('branch')['sales'].sum().to_dict())
print('월평균:', df.groupby('branch')['sales'].mean().to_dict())

In [ ]:
# C05 | 평균표 먼저: 월평균을 계산한 표로 막대를 그립니다.
means = df.groupby('branch', as_index=False)['sales'].mean()
fig_mean = px.bar(means, x='branch', y='sales', color='branch',
                  text_auto=True,                 # 막대 끝에 값
                  labels={**LABELS, 'sales': '월평균 매출 (백만원)'},
                  color_discrete_map=branch_colors,
                  title='1~4월 지점별 월평균 매출')
fig_mean.update_yaxes(range=[0, 160])
fig_mean.show()
print(means)

### 직접 해 보기

**W9-02** 툴팁으로 겹친 점 두 개를 확인합니다.

1. C03의 그림에서 광고비 12·매출 120의 점에 마우스를 올려 툴팁의 지점·월·영업이익을 읽습니다.
2. 범례에서 툴팁에 나온 지점을 한 번 눌러 숨기고, 같은 자리에 남은 점의 툴팁을 읽습니다.
3. 읽은 두 행을 `df[(df['branch'] == …) & (df['month'] == …)]`처럼 두 조건으로 꺼내 툴팁 값과 각각 대조합니다.
4. 범례에서 숨긴 지점을 다시 눌러 되살립니다.

**완료 기준**: 한 자리에 두 행이 있음을 확인했고, 두 툴팁의 값이 원자료 행과 같으며, 숨긴 지점을 되살렸고, 광고비와 매출의 관계에서 원인을 추정하지 않았다.

In [ ]:
# W9-02 — 여기에서 직접 해 봅니다.

## 9.3 Figure 구조와 수정

In [ ]:
# C06 | Figure 읽기: 트레이스 목록과 전체 설정에서 값을 꺼냅니다.
print(type(fig_line).__name__)
print('트레이스 수:', len(fig_line.data))
print('첫 트레이스:', fig_line.data[0].name, fig_line.data[0].y.tolist())
print('제목:', fig_line.layout.title.text)

In [ ]:
# C07 | 두 경로: 선은 update_traces로, 전체는 update_layout으로 고칩니다.
fig_style = go.Figure(fig_line)                         # 복사본
fig_style.update_traces(line_width=2, marker_size=7)    # 모든 트레이스
fig_style.update_traces(line_width=4, marker_size=10,
                        selector={'name': 'A'})         # A만
fig_style.update_layout(title='1~4월 지점별 매출 비교: A지점 강조',
                        legend_title_text='지점', hovermode='x unified')
fig_style.show()

### 직접 해 보기

**W9-03** 아래 시작 코드는 월평균 막대를 그리려다 합계를 냈습니다. 평균표로 고쳐 출력으로 확인하고, `update_traces()`의 `selector`로 A 막대만 골라 테두리를 두릅니다. 테두리는 `marker_line_width=3`과 `marker_line_color='#333333'`으로 정합니다. 제목과 Y축 라벨에는 C05처럼 "월평균"을 넣고(`{**LABELS, …}`), Y축이 0에서 시작하는지 봅니다.

```python
start = df.groupby('branch', as_index=False)['sales'].sum()
fig_w3 = px.bar(start, x='branch', y='sales', color='branch',
                labels=LABELS, color_discrete_map=branch_colors)
fig_w3.show()
```

**완료 기준**: 평균표를 출력해 확인했고, 테두리는 A 막대에만 있으며, 두 지점 색이 그대로이고, 제목·Y축 라벨에 월평균과 단위가 있으며 Y축이 0에서 시작한다.

In [ ]:
# W9-03 — 여기에서 직접 해 봅니다.

## 9.4 탐색 조작과 검수

In [ ]:
# C08 | 화면 조작: 확대·숨기기·복원은 원자료를 바꾸지 않습니다.
fig_explore = go.Figure(fig_style)
fig_explore.update_layout(dragmode='zoom')   # 끌어서 고른 영역을 확대
fig_explore.show()
print('원자료 행 수:', len(df))

In [ ]:
# C09 | 진단용: 월 순서가 섞인 표로 선을 그려 봅니다.
scrambled = df.iloc[[0, 2, 1, 3, 4, 6, 5, 7]]   # 월이 1, 3, 2, 4 순서
fig_unsorted = px.line(scrambled, x='month', y='sales', color='branch',
                       line_dash='branch', markers=True, labels=LABELS,
                       color_discrete_map=branch_colors,
                       title='진단용: 월 순서가 섞인 표로 그린 선')
fig_unsorted.update_xaxes(dtick=1)
fig_unsorted.update_yaxes(range=[0, 160])
fig_unsorted.show()
print('A의 X 순서:', fig_unsorted.data[0].x.tolist())

In [ ]:
# C09B | 정렬 검산: 지점·월 순서로 정렬하고 중복을 확인합니다.
ordered = scrambled.sort_values(['branch', 'month'])
print('월·지점 중복:', ordered.duplicated(['branch', 'month']).any())
fig_sorted = px.line(ordered, x='month', y='sales', color='branch',
                     line_dash='branch', markers=True)
print('정렬 뒤 A의 X 순서:', fig_sorted.data[0].x.tolist())

In [ ]:
# C10 | 최종 차트: 질문 하나에 답하는 제목·주석·출처를 붙입니다.
wide = df.pivot(index='month', columns='branch', values='sales')
gap = wide['A'] - wide['B']                  # 월별 A-B 격차
print('월별 A-B 격차:', gap.tolist())
a4, b4 = wide.loc[4, 'A'], wide.loc[4, 'B']
final_fig = go.Figure(fig_style)
final_fig.update_layout(
    title=f'1~4월 지점별 매출: 4월 격차는 {a4 - b4}백만원')
final_fig.add_annotation(x=4, y=a4, text=f'4월 A {a4} / B {b4}',
                         showarrow=True, ax=-45, ay=-45)
final_fig.add_annotation(text='자료: 교육용 가상 자료(월 합계)',
                         xref='paper', yref='paper', x=0.01, y=0.03,
                         xanchor='left', showarrow=False,
                         font_size=12, font_color='#666666')
final_fig.show()

## 9.5 HTML 저장과 공유

In [ ]:
# C11 | HTML 저장: 인터넷 없이 열리는 파일로 저장하고 확인합니다.
from pathlib import Path

output_dir = Path('ch09_output')
output_dir.mkdir(exist_ok=True)
html_path = output_dir / 'ch09_final.html'
final_fig.write_html(html_path, include_plotlyjs=True, full_html=True)
size_mb = html_path.stat().st_size / 1_000_000   # 바이트 → MB
print('크기:', round(size_mb, 1), 'MB')
print('Plotly.js 포함:', size_mb > 1)            # 'cdn'이면 약 0.01MB

### 직접 해 보기

**W9-04** 탐색하는 차트를 HTML로 완성합니다.

1. `w4 = go.Figure(final_fig)`로 복사하고, `gap`의 3월 격차로 3월 A지점 점에 주석을 더하되 `ax`·`ay`로 4월 주석과 겹치지 않게 놓습니다.
2. `ch09_output/W9-04.html`로 저장하고 브라우저에서 새로 엽니다.
3. 4월 툴팁, B 숨기기와 켜기, 확대와 되돌리기를 해 봅니다.
4. 두 주석이 보여 주는 변화를 관찰 한 문장으로, 원인을 알려면 더 확인할 자료를 한 문장으로 씁니다(8.5절의 추가 확인).

**완료 기준**: 두 주석이 겹치지 않고, 새로 연 HTML에서 네 조작이 되며, 관찰 문장에 기간·대상·수치·단위가 있고, 원인 대신 확인할 자료를 적었다.

In [ ]:
# W9-04 — 여기에서 직접 해 봅니다.

## 9.6 해석과 점검

## 직접 해 보기 풀이

먼저 위의 연습 칸에서 직접 해 본 뒤에 봅니다. 풀이는 한 가지 예이고, 완료 기준을 만족하면 다른 코드도 됩니다.

### W9-01

In [ ]:
# W9-01 | 풀이: C02를 새 이름으로 복사해 영업이익 선 그래프로 바꿉니다.
fig_profit = px.line(df, x='month', y='profit', color='branch',
                     line_dash='branch', markers=True, labels=LABELS,
                     color_discrete_map=branch_colors,
                     title='1~4월 지점별 영업이익')
fig_profit.update_xaxes(dtick=1)
fig_profit.update_yaxes(range=[0, 30])          # 단위는 LABELS의 '영업이익 (백만원)'
fig_profit.show()

In [ ]:
# 3. 툴팁에서 읽은 A지점 최고 월·값을 원자료와 대조
print('점 개수:', [len(trace.y) for trace in fig_profit.data])
print(df[(df['branch'] == 'A') & (df['profit'] == df.loc[df['branch'] == 'A', 'profit'].max())])
assert [len(trace.y) for trace in fig_profit.data] == [4, 4]
# 답(예): 툴팁에서 A지점 3월 영업이익 25백만원을 읽었고, df의 같은 행과 같다.

### W9-02

In [ ]:
# W9-02 | 풀이: 광고비 12·매출 120 자리에 겹친 두 점을 원자료로 대조합니다.
# 1. C03의 그림에서 그 점에 마우스를 올리면 위에 그려진 점의 툴팁이 뜹니다(B지점 4월).
# 2. 범례에서 B를 한 번 눌러 숨기면 같은 자리에 A지점 2월 점이 남습니다.
# 3. 읽은 두 행을 두 조건으로 꺼내 툴팁과 대조합니다.
print(df[(df['branch'] == 'B') & (df['month'] == 4)])
print(df[(df['branch'] == 'A') & (df['month'] == 2)])
same_spot = df[(df['marketing'] == 12) & (df['sales'] == 120)]
assert len(same_spot) == 2
# 4. 범례의 B를 다시 눌러 되살립니다.
# 답(예): 광고비 12·매출 120백만원 자리에 B지점 4월(영업이익 18)과 A지점 2월(영업이익 15) 두 행이 있다.
#   두 점이 같은 자리에 있다고 광고비가 매출을 정한다고 볼 수는 없다.

### W9-03

In [ ]:
# W9-03 | 풀이: 합계를 낸 시작 코드를 평균표로 고치고 A 막대에만 테두리를 두릅니다.
means_w3 = df.groupby('branch', as_index=False)['sales'].mean()   # sum() → mean()
print(means_w3)
fig_w3 = px.bar(means_w3, x='branch', y='sales', color='branch',
                labels={**LABELS, 'sales': '월평균 매출 (백만원)'},
                color_discrete_map=branch_colors,
                title='1~4월 지점별 월평균 매출')
fig_w3.update_traces(marker_line_width=3, marker_line_color='#333333',
                     selector={'name': 'A'})                     # A 막대만
fig_w3.update_yaxes(range=[0, 160])                              # 0에서 시작
fig_w3.show()

In [ ]:
print([(trace.name, trace.marker.line.width, trace.marker.color) for trace in fig_w3.data])
assert means_w3['sales'].tolist() == [125, 95]
assert fig_w3.data[0].marker.line.width == 3 and fig_w3.data[1].marker.line.width is None

### W9-04

In [ ]:
# W9-04 | 풀이: 최종 차트에 3월 주석을 더해 HTML로 저장합니다.
from pathlib import Path

w4 = go.Figure(final_fig)
a3 = wide.loc[3, 'A']
w4.add_annotation(x=3, y=a3, text=f'3월 격차 {gap.loc[3]}',
                  showarrow=True, ax=-70, ay=35)           # 아래로 — 4월 주석(ax=-45, ay=-45)·제목과 떨어지게
output_dir = Path('ch09_output')
output_dir.mkdir(exist_ok=True)
w4.write_html(output_dir / 'W9-04.html', include_plotlyjs=True, full_html=True)
w4.show()
print('주석 수:', len(w4.layout.annotations), '| 3월 격차:', gap.loc[3], '| 4월 격차:', gap.loc[4])
assert gap.loc[3] == 50 and gap.loc[4] == 10
# 3. 브라우저에서 ch09_output/W9-04.html을 새로 열어 4월 툴팁, B 숨기기와 켜기, 확대와 되돌리기를 해 봅니다.
# 4. 관찰(예): 1~4월 A지점과 B지점의 매출 격차는 3월 50백만원에서 4월 10백만원으로 줄었다.
#    추가 확인(예): 격차가 줄어든 까닭을 알려면 두 지점의 4월 고객 수나 판촉 기록을 더 본다.